# 🎮 Polymarket — Top Holders dos mercados resolvidos de League of Legends

Notebook que varre mercados de LoL fechados no Polymarket nos últimos N dias, identifica o lado vencedor, e ranqueia os top holders por P&L.

**Objetivos:**
- 🎯 **Copy-trading research** — quem acerta consistentemente em mercados LoL
- 🐳 **Whale tracking** — wallets recorrentes em múltiplos mercados
- 📊 **Análise de P&L** — preço médio de entrada vs resolução em \$1

## Como usar

1. Ajuste os parâmetros na próxima célula (sliders/forms à direita)
2. Execute todas as células: `Runtime → Run all` (Ctrl+F9)
3. Veja gráficos interativos no final
4. Faça download dos CSVs/JSON gerados

⚠️ **Sandbox:** os endpoints da Data API são best-effort. Se gráficos vierem vazios, ative `DEBUG=True` na célula de parâmetros e veja as chamadas raw no console.

## 1. Setup

In [ ]:
%pip install -q plotly pandas requests

import json, time, sys, re
from datetime import datetime, timezone, timedelta
import requests
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display, Markdown, HTML

print('Setup OK — Python', sys.version.split()[0], '| pandas', pd.__version__)

## 2. Parâmetros

In [ ]:
#@title 🎛️ Configuração { display-mode: "form" }
#@markdown ### Janela e escopo
DAYS = 30 #@param {type:"slider", min:7, max:365, step:1}
TOP_N_PER_MARKET = 10 #@param {type:"slider", min:5, max:50, step:5}
TAG_OVERRIDE = "" #@param {type:"string"}
MARKETS_OVERRIDE = "" #@param {type:"string"}
#@markdown _Deixe vazio para auto-detectar tag de LoL. `MARKETS_OVERRIDE` aceita slugs separados por vírgula._

#@markdown ### Performance
COMPUTE_PNL = True #@param {type:"boolean"}
RATE_LIMIT_MS = 100 #@param {type:"slider", min:50, max:1000, step:50}
DEBUG = False #@param {type:"boolean"}

#@markdown ### Filtros para visualização
MIN_PNL_FOR_PLOTS = 0 #@param {type:"number"}
TOP_GLOBAL_N = 20 #@param {type:"slider", min:5, max:100, step:5}

print(f'DAYS={DAYS}  TOP={TOP_N_PER_MARKET}  TAG={TAG_OVERRIDE or "(auto)"}  PNL={COMPUTE_PNL}  RATE={RATE_LIMIT_MS}ms')

## 3. Cliente das APIs (Gamma + Data)

In [ ]:
GAMMA_API = 'https://gamma-api.polymarket.com'
DATA_API = 'https://data-api.polymarket.com'
# Real Polymarket tag slugs (from URL inspection: polymarket.com/esports/league-of-legends/lec/...)
LOL_COMPETITION_TAGS = ('lec', 'lck', 'lpl', 'lcs', 'lcp', 'cblol', 'lol-worlds', 'worlds', 'msi', 'first-stand', 'league-of-legends')
DEFAULT_TAG_CANDIDATES = LOL_COMPETITION_TAGS + ('esports',)
# Every LoL market slug starts with 'lol-' (e.g. lol-mkoi-kc-2026-05-09) — most reliable match
LOL_SLUG_PREFIXES = ('lol-',)
LOL_KEYWORDS = ('league of legends', 'lck', 'lpl', 'lec', 'lcs', 'worlds', 'msi')
MAX_PAGES = 30  # hard cap on pagination (= 3000 markets max per fetch)
LOL_KEYWORD_MIN_RATIO = 0.30  # first page <30% LoL match = tag silently ignored


def sanitize_text(text):
    """Strip control chars; markets are user-generated content."""
    if not text: return ''
    text = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]', '', text)
    return text[:200] + '...' if len(text) > 200 else text


def _has_lol_slug(market):
    slug = (market.get('slug') or '').lower()
    return any(slug.startswith(p) for p in LOL_SLUG_PREFIXES)


def _has_lol_keyword(market):
    """Match if slug prefix OR text keyword (slug works for non-English too)."""
    if _has_lol_slug(market): return True
    text = (market.get('question', '') + ' ' + (market.get('description') or '')).lower()
    return any(kw in text for kw in LOL_KEYWORDS)


class API:
    def __init__(self, rate_ms=100, debug=False):
        self.s = requests.Session()
        self.s.headers['User-Agent'] = 'polymarket-skills-colab'
        self.rate = rate_ms / 1000.0
        self.debug = debug
        self._last = 0.0

    def get(self, url, params=None):
        wait = self.rate - (time.monotonic() - self._last)
        if wait > 0: time.sleep(wait)
        last_err = None
        for attempt in range(3):
            try:
                r = self.s.get(url, params=params or {}, timeout=30)
                self._last = time.monotonic()
                if self.debug: print(f'  GET {r.url} -> {r.status_code}')
                if r.status_code == 429:
                    time.sleep(2 ** attempt); continue
                r.raise_for_status()
                return r.json()
            except requests.exceptions.RequestException as e:
                last_err = e
                if attempt == 2: raise
                time.sleep(2 ** attempt)
        if last_err: raise last_err


api = API(rate_ms=RATE_LIMIT_MS, debug=DEBUG)
print('API client ready')

## 4. Lógica de discovery, winner detection e P&L

In [ ]:
def fetch_markets_by_tag(tag, after_iso):
    """Tag-filtered fetch with sanity check + MAX_PAGES cap."""
    out = []
    for page_num in range(MAX_PAGES):
        offset = page_num * 100
        page = api.get(f'{GAMMA_API}/markets', params={
            'tag_slug': tag, 'closed': 'true', 'limit': 100, 'offset': offset,
            'order': 'endDate', 'ascending': 'false',
        })
        if not isinstance(page, list) or not page: break
        # First-page sanity: API silently returns ALL markets if tag is unknown.
        # Use slug-prefix only (`lol-`) — keyword match has too many false positives
        # because short tokens like 'lec' hit 'select', 'lecture', etc.
        if page_num == 0:
            matches = sum(1 for m in page if _has_lol_slug(m))
            ratio = matches / len(page)
            if ratio < LOL_KEYWORD_MIN_RATIO:
                sample = ', '.join((m.get('slug') or '?')[:30] for m in page[:5])
                print(f"  WARN: tag {tag!r} looks ignored ({matches}/{len(page)}={ratio:.0%} 'lol-' prefix) — aborting")
                print(f'  first 5 slugs: {sample}')
                return []
        keep, stop = [], False
        for m in page:
            end = m.get('endDate') or ''
            if not end: continue
            if end < after_iso: stop = True; break
            keep.append(m)
        out.extend(keep)
        if stop or len(page) < 100: break
    return out


def fetch_markets_by_search(query, after_iso):
    """Text-search fallback with client-side LoL filter."""
    out = []
    for page_num in range(MAX_PAGES):
        offset = page_num * 100
        page = api.get(f'{GAMMA_API}/markets', params={
            'q': query, 'closed': 'true', 'limit': 100, 'offset': offset,
            'order': 'endDate', 'ascending': 'false',
        })
        if not isinstance(page, list) or not page: break
        keep, stop = [], False
        for m in page:
            end = m.get('endDate') or ''
            if not end: continue
            if end < after_iso: stop = True; break
            if _has_lol_keyword(m): keep.append(m)
        out.extend(keep)
        if stop or len(page) < 100: break
    return out


def _event_has_lol_tag(event):
    tags = event.get('tags') or []
    if isinstance(tags, list):
        for t in tags:
            slug = (t.get('slug') if isinstance(t, dict) else str(t)).lower()
            if slug in LOL_COMPETITION_TAGS or slug.startswith('lol-'):
                return True
    return _has_lol_slug(event)


def fetch_events_by_tag(tag, after_iso):
    """Query /events for one LoL competition tag, return flattened market dicts."""
    out = []
    for page_num in range(MAX_PAGES):
        offset = page_num * 100
        page = api.get(f'{GAMMA_API}/events', params={
            'tag_slug': tag, 'closed': 'true', 'limit': 100, 'offset': offset,
            'order': 'endDate', 'ascending': 'false',
        })
        if not isinstance(page, list) or not page: break
        if page_num == 0:
            tagged = sum(1 for ev in page if _event_has_lol_tag(ev))
            ratio = tagged / len(page)
            if ratio < LOL_KEYWORD_MIN_RATIO:
                sample = ', '.join((ev.get('slug') or '?')[:30] for ev in page[:5])
                print(f"  WARN: /events tag {tag!r} looks ignored ({tagged}/{len(page)}={ratio:.0%} LoL-tagged) — aborting")
                print(f'  first 5 event slugs: {sample}')
                return []
        keep_events, stop = [], False
        for ev in page:
            end = ev.get('endDate') or ''
            if not end: continue
            if end < after_iso: stop = True; break
            keep_events.append(ev)
        for ev in keep_events:
            for m in (ev.get('markets') or []):
                if not m.get('endDate'):
                    m['endDate'] = ev.get('endDate', '')
                m.setdefault('eventSlug', ev.get('slug', ''))
                out.append(m)
        if stop or len(page) < 100: break
    return out


def fetch_markets_by_slug_prefix(after_iso):
    """Most reliable: scan recent closed markets, client-filter by 'lol-' slug prefix."""
    out = []
    for page_num in range(MAX_PAGES):
        offset = page_num * 100
        page = api.get(f'{GAMMA_API}/markets', params={
            'closed': 'true', 'limit': 100, 'offset': offset,
            'order': 'endDate', 'ascending': 'false',
        })
        if not isinstance(page, list) or not page: break
        keep, stop = [], False
        for m in page:
            end = m.get('endDate') or ''
            if not end: continue
            if end < after_iso: stop = True; break
            if _has_lol_slug(m): keep.append(m)
        out.extend(keep)
        if stop or len(page) < 100: break
    return out


def discover_markets(tag_override, days, explicit_slugs):
    after_iso = (datetime.now(timezone.utc) - timedelta(days=days)).isoformat()
    if explicit_slugs:
        out = []
        for slug in explicit_slugs:
            try:
                page = api.get(f'{GAMMA_API}/markets', params={'slug': slug})
                if isinstance(page, list): out.extend(page)
            except Exception as e:
                print(f'WARN: {slug}: {e}')
        return ('explicit', out)

    # PRIMARY: query /events for ALL LoL competitions and merge unique markets
    if not tag_override:
        print('Querying /events for all LoL competitions')
        all_markets, seen, tags_with = [], set(), []
        for tag in LOL_COMPETITION_TAGS:
            try:
                ms = fetch_events_by_tag(tag, after_iso)
            except Exception as e:
                if DEBUG: print(f'  /events tag {tag} failed: {e}')
                continue
            if not ms: continue
            new = 0
            for m in ms:
                cid = m.get('conditionId') or m.get('condition_id') or ''
                if cid and cid not in seen:
                    seen.add(cid); all_markets.append(m); new += 1
            print(f'  /events tag={tag!r}: {len(ms)} markets ({new} new)')
            if new > 0: tags_with.append(tag)
        if all_markets:
            return (f"events:{','.join(tags_with)}", all_markets)
        print('No /events tag yielded; falling through to /markets path')

    candidates = (tag_override,) if tag_override else DEFAULT_TAG_CANDIDATES
    for slug in candidates:
        if not slug: continue
        print(f'Trying /markets tag_slug={slug!r}')
        try:
            ms = fetch_markets_by_tag(slug, after_iso)
            print(f'  -> {len(ms)} market(s)')
            if ms: return (slug, ms)
        except Exception as e:
            if DEBUG: print(f'  tag {slug} failed: {e}')

    print('Falling back to text search')
    try:
        ms = fetch_markets_by_search('league of legends', after_iso)
        if ms: return ('text:lol', ms)
    except Exception as e:
        print(f'text search failed: {e}')

    print("Falling back to slug-prefix scan ('lol-')")
    try:
        return ('slug-prefix:lol-', fetch_markets_by_slug_prefix(after_iso))
    except Exception as e:
        print(f'slug-prefix scan failed: {e}')
        return ('error', [])


def determine_winner(market):
    try:
        outcomes = json.loads(market.get('outcomes', '[]'))
        prices = [float(p) for p in json.loads(market.get('outcomePrices', '[]'))]
        token_ids = json.loads(market.get('clobTokenIds', '[]'))
    except Exception:
        return None
    if not (outcomes and prices and token_ids and len(outcomes) == len(prices) == len(token_ids)):
        return None
    idx = max(range(len(prices)), key=lambda i: prices[i])
    if prices[idx] < 0.99: return None
    return {'idx': idx, 'outcome': sanitize_text(outcomes[idx]), 'token_id': str(token_ids[idx]), 'price': prices[idx]}


def fetch_holders(condition_id, token_id, limit):
    """Real /holders shape: [{token, holders: [{proxyWallet, amount, name, ...}]}, ...]
    Find the entry whose token matches the winning token_id, return its holders.
    """
    try:
        data = api.get(f'{DATA_API}/holders', params={'market': condition_id, 'limit': limit})
    except Exception:
        return []
    if not isinstance(data, list):
        if DEBUG: print(f'    [holders] unexpected response type: {type(data).__name__}')
        return []
    target = str(token_id)
    for entry in data:
        if not isinstance(entry, dict): continue
        if str(entry.get('token') or '') != target: continue
        normalized = []
        for h in (entry.get('holders') or []):
            if not isinstance(h, dict): continue
            addr = h.get('proxyWallet') or h.get('user') or h.get('address') or ''
            try: shares = float(h.get('amount') or h.get('size') or h.get('balance') or 0)
            except (TypeError, ValueError): shares = 0.0
            if not addr or shares <= 0: continue
            normalized.append({
                'address': addr.lower(),
                'shares': shares,
                'name': (h.get('name') or h.get('pseudonym') or '').strip(),
            })
        normalized.sort(key=lambda x: x['shares'], reverse=True)
        return normalized[:limit]
    if DEBUG:
        tokens_seen = [str(e.get('token','?'))[:20] for e in data if isinstance(e, dict)]
        print(f'    [holders] target {target[:20]}... not in response (tokens: {tokens_seen})')
    return []

def fetch_trades(condition_id, address):
    attempts = (
        ('/trades', {'market': condition_id, 'user': address}),
        ('/trades', {'market': condition_id, 'maker': address}),
        ('/trades', {'market': condition_id, 'address': address}),
    )
    for path, params in attempts:
        try:
            data = api.get(f'{DATA_API}{path}', params=params)
        except Exception:
            continue
        items = data if isinstance(data, list) else (data or {}).get('data') or []
        if items: return items
    return []


def estimate_pnl(trades, win_token, current):
    bs, bc, real, n = 0.0, 0.0, 0.0, 0
    for t in trades:
        tok = str(t.get('tokenId') or t.get('token_id') or t.get('asset_id') or '')
        if tok and tok != str(win_token): continue
        side = (t.get('side') or t.get('type') or '').upper()
        try:
            p = float(t.get('price', 0)); s = float(t.get('size', 0))
        except Exception:
            continue
        if s <= 0 or p <= 0: continue
        if side in ('BUY', 'BID'):
            bs += s; bc += p * s; n += 1
        elif side in ('SELL', 'ASK'):
            avg = bc / bs if bs > 0 else 0
            real += (p - avg) * s
            bs -= s; bc -= avg * s; n += 1
    avg = bc / bs if bs > 0 else 0
    unr = (1.0 - avg) * current if current > 0 and avg > 0 else 0
    return {
        'avg_entry': round(avg, 4),
        'realized': round(real, 2),
        'unrealized': round(unr, 2),
        'total_pnl': round(real + unr, 2),
        'n_trades': n,
    }


print('Helpers loaded')

## 5. Run pipeline

In [ ]:
explicit = [s.strip() for s in MARKETS_OVERRIDE.split(',') if s.strip()] if MARKETS_OVERRIDE else None
tag_used, raw_markets = discover_markets(TAG_OVERRIDE or None, DAYS, explicit)
print(f'\nDiscovered {len(raw_markets)} closed market(s) via {tag_used!r}\n')

rows = []
markets_summary = []
for raw in raw_markets:
    slug = raw.get('slug', '?')
    winner = determine_winner(raw)
    if not winner:
        markets_summary.append({'slug': slug, 'status': 'no_resolution', 'n_holders': 0})
        continue
    cond = raw.get('conditionId') or raw.get('condition_id') or ''
    print(f'  {slug}: {winner["outcome"]!r} won (price={winner["price"]:.3f})')
    holders = fetch_holders(cond, winner['token_id'], TOP_N_PER_MARKET)
    print(f'    -> {len(holders)} holder(s)')
    markets_summary.append({
        'slug': slug, 'status': 'ok', 'winner': winner['outcome'],
        'end_date': raw.get('endDate', ''), 'n_holders': len(holders),
    })
    for h in holders:
        row = {
            'market_slug': slug,
            'question': sanitize_text(raw.get('question', '')),
            'end_date': raw.get('endDate', ''),
            'winning_outcome': winner['outcome'],
            'address': h['address'],
            'name': h.get('name', ''),
            'shares_at_resolution': round(h['shares'], 2),
        }
        if COMPUTE_PNL:
            trades = fetch_trades(cond, h['address'])
            row.update(estimate_pnl(trades, winner['token_id'], h['shares']))
        rows.append(row)

df = pd.DataFrame(rows)
summary_df = pd.DataFrame(markets_summary)
print(f'\n✓ {len(df)} holder rows across {summary_df[summary_df["status"] == "ok"].shape[0] if not summary_df.empty else 0} resolved markets')
if not summary_df.empty: display(summary_df)

## 6. 📊 Visualizações

Se algum gráfico ficar vazio, a Data API provavelmente mudou os shapes dos endpoints. Ative `DEBUG=True` na célula 2 e re-execute pra ver as chamadas raw.

### 6.1 Top wallets globais (cross-market)

In [ ]:
if df.empty:
    display(Markdown('⚠️ **Nenhum dado pra plotar.** Verifique parâmetros e/ou ative `DEBUG=True`.'))
elif 'total_pnl' in df.columns:
    glob = df.groupby('address').agg(
        total_pnl=('total_pnl', 'sum'),
        n_winning=('market_slug', 'nunique'),
        total_shares=('shares_at_resolution', 'sum'),
    ).reset_index().sort_values(['total_pnl', 'n_winning'], ascending=[False, False]).head(TOP_GLOBAL_N)
    glob['short_addr'] = glob['address'].str[:8] + '...' + glob['address'].str[-4:]
    fig = px.bar(
        glob, x='short_addr', y='total_pnl', color='n_winning',
        hover_data=['address', 'total_shares', 'n_winning'],
        labels={'short_addr': 'Wallet', 'total_pnl': 'Total P&L (USD)', 'n_winning': 'Mercados vencidos'},
        title=f'Top {TOP_GLOBAL_N} wallets globais por P&L total',
        color_continuous_scale='Viridis',
    )
    fig.update_layout(xaxis_tickangle=-45, height=500)
    fig.show()
else:
    glob = df.groupby('address').agg(
        total_shares=('shares_at_resolution', 'sum'),
        n_winning=('market_slug', 'nunique'),
    ).reset_index().sort_values('total_shares', ascending=False).head(TOP_GLOBAL_N)
    glob['short_addr'] = glob['address'].str[:8] + '...' + glob['address'].str[-4:]
    fig = px.bar(glob, x='short_addr', y='total_shares', color='n_winning',
                 title=f'Top {TOP_GLOBAL_N} wallets globais por shares vencedoras (P&L desativado)')
    fig.update_layout(xaxis_tickangle=-45, height=500)
    fig.show()

### 6.2 Top holders por mercado

In [ ]:
if not df.empty:
    n_markets = df['market_slug'].nunique()
    if n_markets <= 12:
        for slug in df['market_slug'].unique():
            sub = df[df['market_slug'] == slug].head(TOP_N_PER_MARKET).copy()
            sub['short_addr'] = sub['address'].str[:8] + '...' + sub['address'].str[-4:]
            title = (sub['question'].iloc[0][:80] + '...') if len(sub['question'].iloc[0]) > 80 else sub['question'].iloc[0]
            color_col = 'total_pnl' if 'total_pnl' in sub.columns else None
            fig = px.bar(
                sub, x='short_addr', y='shares_at_resolution', color=color_col,
                title=f'<b>{title}</b><br><sub>vencedor: {sub["winning_outcome"].iloc[0]}</sub>',
                labels={'short_addr': 'Wallet', 'shares_at_resolution': 'Shares na resolução', 'total_pnl': 'P&L (USD)'},
                color_continuous_scale='RdYlGn' if color_col else None,
            )
            fig.update_layout(xaxis_tickangle=-45, height=350)
            fig.show()
    else:
        display(Markdown(f'_({n_markets} mercados — pulando per-market plots; veja agregado e heatmap abaixo.)_'))

### 6.3 Skill scope: avg entry vs P&L

Quanto mais à esquerda + alto, mais skill (entrou cedo, lucrou alto). Pontos no extremo direito são wallets que entraram tarde quando o resultado já estava quase decidido.

In [ ]:
if 'total_pnl' in df.columns and not df.empty:
    sd = df[(df['avg_entry'] > 0) & (df['total_pnl'] != 0)].copy()
    if not sd.empty:
        sd['short_addr'] = sd['address'].str[:10] + '...'
        fig = px.scatter(
            sd, x='avg_entry', y='total_pnl',
            size='shares_at_resolution', color='market_slug',
            hover_data=['address', 'market_slug', 'n_trades'],
            title='Entry price vs P&L por holder',
            labels={'avg_entry': 'Avg entry price (USD)', 'total_pnl': 'P&L (USD)'},
        )
        fig.add_hline(y=0, line_dash='dash', line_color='gray')
        fig.add_vline(x=0.5, line_dash='dot', line_color='gray', annotation_text='50% odds')
        fig.update_layout(height=500)
        fig.show()
    else:
        display(Markdown('_(Sem dados de P&L com entry > 0 e P&L ≠ 0.)_'))

### 6.4 Heatmap: wallets em múltiplos mercados

Só mostra wallets que aparecem em 2+ mercados — sinal de comportamento sistemático.

In [ ]:
if not df.empty and df['address'].nunique() <= 100:
    metric = 'total_pnl' if 'total_pnl' in df.columns else 'shares_at_resolution'
    heat = df.pivot_table(index='address', columns='market_slug',
                          values=metric, aggfunc='sum', fill_value=0)
    heat = heat[(heat != 0).sum(axis=1) >= 2]
    if not heat.empty:
        heat.index = heat.index.str[:10] + '...'
        fig = go.Figure(data=go.Heatmap(
            z=heat.values, x=heat.columns, y=heat.index,
            colorscale='RdYlGn', zmid=0,
            colorbar=dict(title=metric.replace('_', ' ').title()),
        ))
        fig.update_layout(
            title=f'Cross-market heatmap ({metric})',
            xaxis_tickangle=-45, height=max(300, 25 * len(heat)),
        )
        fig.show()
    else:
        display(Markdown('_Nenhuma wallet apareceu em 2+ mercados na janela atual._'))

### 6.5 Distribuição de P&L

In [ ]:
if 'total_pnl' in df.columns and not df.empty:
    fig = px.histogram(
        df, x='total_pnl', nbins=40,
        title='Distribuição de P&L entre top holders',
        labels={'total_pnl': 'P&L (USD)'},
        color_discrete_sequence=['#3b82f6'],
    )
    fig.add_vline(x=0, line_dash='dash', line_color='red', annotation_text='Break-even')
    fig.add_vline(x=df['total_pnl'].median(), line_dash='dot', line_color='green',
                  annotation_text=f'Median ${df["total_pnl"].median():.0f}')
    fig.update_layout(height=400)
    fig.show()
    print(f'Median P&L: ${df["total_pnl"].median():>10.2f}')
    print(f'Mean P&L:   ${df["total_pnl"].mean():>10.2f}')
    print(f'Top 10% cutoff: ${df["total_pnl"].quantile(0.9):>6.2f}')
    print(f'Bottom 10%:     ${df["total_pnl"].quantile(0.1):>6.2f}')

## 7. 📋 Tabela completa

Tabela ordenada para inspeção e copy-paste.

In [ ]:
if not df.empty:
    sort_col = 'total_pnl' if 'total_pnl' in df.columns else 'shares_at_resolution'
    display_df = df.sort_values(sort_col, ascending=False).reset_index(drop=True)
    display(display_df.head(50).style.background_gradient(
        subset=[c for c in [sort_col, 'avg_entry'] if c in display_df.columns],
        cmap='RdYlGn',
    ))

## 8. 💾 Export

Salva CSV + JSON no ambiente Colab e dispara download para sua máquina.

In [ ]:
ts = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M')
csv_path = f'lol_top_holders_{ts}.csv'
json_path = f'lol_top_holders_{ts}.json'

if not df.empty:
    df.to_csv(csv_path, index=False)
    report = {
        'scanned_at': datetime.now(timezone.utc).isoformat(),
        'window_days': DAYS,
        'tag_slug_used': tag_used,
        'markets_summary': markets_summary,
        'rows': df.to_dict(orient='records'),
    }
    with open(json_path, 'w') as f:
        json.dump(report, f, indent=2, ensure_ascii=False)
    print(f'Salvos:\n  {csv_path}\n  {json_path}')
    try:
        from google.colab import files
        files.download(csv_path)
        files.download(json_path)
    except Exception:
        print('(Não está no Colab — arquivos disponíveis no diretório atual.)')
else:
    print('Nada pra exportar — DataFrame vazio.')

---

## 🛠️ Troubleshooting

| Sintoma | Causa provável | Fix |
|---|---|---|
| `Discovered 0 closed market(s)` | tag_slug de LoL não existe ou janela vazia | Aumente `DAYS` para 90/180; ou passe `TAG_OVERRIDE=esports` |
| `0 holder(s)` em todos os mercados | endpoint `/holders` mudou shape | Ative `DEBUG=True`, role pro topo, veja a chamada raw e ajuste `fetch_holders` |
| `n_trades=0` em todos os holders | endpoint `/trades` mudou shape | Mesmo: ative DEBUG, ajuste `fetch_trades`. Como workaround, marque `COMPUTE_PNL=False` |
| 403 / 429 | Rate limit ou bloqueio | Aumente `RATE_LIMIT_MS` para 500-1000 |
| Heatmap vazio | Nenhuma wallet em 2+ mercados | Aumente janela ou top N |